# Kafka Demo

## Important: Connect to Kafka Broker Server FIRST

**Before running any code in this notebook**, establish an SSH tunnel to the Kafka server:

```
ssh -L 9092:localhost:9092 tunnel@128.2.220.123 -NT
```

Password: `mlip-kafka` (from the Canvas / 17445 Pages Kafka and API Configuration page).

Leave that terminal open. It will not print anything. Then Kafka looks like it is running on your laptop at `localhost:9092`.

**Verify your connection is active:**
```bash
kcat -b localhost:9092 -L
```

**To kill the connection when done:**
```
lsof -ti:9092 | xargs kill -9
```

---

## Setup

```
python -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```


In [1]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer, TopicPartition
from typing import Dict, Any

# Unique identifier so this topic does not collide with other students.
# Replace with your Andrew ID if it is different.
andrew_id = "helenlwang"
topic = f"lab01-{andrew_id}"
print(f"Topic: {topic}")

Topic: lab01-helenlwang


### Producer Mode -> Writes Data to Broker

In [ ]:
# Schema for messages. Cities can be changed; three cities are enough.
def make_city_data(city: str, temperature_f: str) -> Dict[str, Any]:
    return {
        "city": city,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperature_f": temperature_f,  # temperature in fahrenheit
    }

In [ ]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html
#
# bootstrap_servers is the SSH-tunneled local port, not 128.2.220.123:9092.
# Kafka stores bytes, so serialize Python dict -> JSON string -> UTF-8 bytes.

producer = KafkaProducer(
    bootstrap_servers=["localhost:9092"],
    value_serializer=lambda m: dumps(m).encode("utf-8"),
)

In [1]:
# Add a few more cities as (city, temperature_f) pairs
cities = [("Pittsburgh", 64), ("Shanghai", 82), ("San Francisco", 68)]

# 20 messages at 0.5s takes ~10 seconds and gives a wider offset range to replay.
NUM_MESSAGES = 20

print("Writing to Kafka Broker")
assigned_offsets = []
for i in range(NUM_MESSAGES):
    city, temperature_f = cities[randint(0, len(cities) - 1)]  # random selection
    data = make_city_data(city, temperature_f)
    future = producer.send(topic=topic, value=data)
    # Kafka assigns every message an offset: its position in the partition's log.
    assigned_offsets.append(future.get(timeout=10).offset)
    sleep(0.5)

producer.flush()
print(f"Data written to topic: {topic}")
print(f"Offsets assigned in this run: {assigned_offsets[0]} .. {assigned_offsets[-1]}")

Writing to Kafka Broker
Data written to topic: lab01-helenlwang
Offsets assigned in this run: 0 .. 19


### Consumer Mode -> Reads Data from Broker

In [1]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html
#
# auto_offset_reset tradeoffs:
#   'earliest' — if this consumer group has no committed offset, start at the log beginning
#   'latest'   — if no committed offset, skip history and wait for brand-new messages
#   'none'     — if no committed offset, raise an error instead of guessing
# It only chooses between the TWO ENDS, and only when there is no valid committed offset.
# consumer_timeout_ms stops the loop after idle time so this cell does not hang forever.

consumer = KafkaConsumer(
    topic,
    bootstrap_servers=["localhost:9092"],
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    auto_commit_interval_ms=1000,
    consumer_timeout_ms=5000,
    group_id=f"{topic}-notebook-{datetime.now().strftime('%H%M%S')}",
)

print("Reading Kafka Broker")
for message in consumer:
    message_str = message.value.decode("utf-8")
    message_dict = loads(message_str)
    print(f"offset {message.offset}: {message_dict}")
    os.system(f"echo {message_str} >> kafka_log.csv")

Reading Kafka Broker
offset 0: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:17', 'temperature_f': 82}
offset 1: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 18:45:18', 'temperature_f': 64}
offset 2: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 18:45:18', 'temperature_f': 64}
offset 3: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:19', 'temperature_f': 82}
offset 4: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:19', 'temperature_f': 82}
offset 5: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 18:45:20', 'temperature_f': 64}
offset 6: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 18:45:20', 'temperature_f': 64}
offset 7: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 18:45:21', 'temperature_f': 64}
offset 8: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 18:45:22', 'temperature_f': 64}
offset 9: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:22', 'temperature_f': 82}
offset 10: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:23', 'temperature_f': 82

### Reading From the Middle of the Log

`auto_offset_reset` only picks between the two *ends* of the log -- `earliest` (the log
start offset) and `latest` (the high watermark) -- and Kafka only applies it when a
consumer has no valid committed offset. It cannot start you anywhere in between.

Real consumers need that middle ground: "replay the last 50 messages", "resume 200
messages before where we crashed". For those you use `seek()`.

The cell below does this in one pass: find the two ends of the valid range, then read
from a few different offsets spread across it.


In [1]:
# assign() rather than subscribe(): it hands us a specific partition immediately,
# so seek() works without waiting for a consumer-group rebalance.

explorer = KafkaConsumer(
    bootstrap_servers=["localhost:9092"],
    enable_auto_commit=False,   # don't move any committed offset while exploring
    group_id=None,              # not part of a group: nothing is committed
)
tp = TopicPartition(topic, 0)   # our lab topic has a single partition
explorer.assign([tp])

first_offset = explorer.beginning_offsets([tp])[tp]
next_offset = explorer.end_offsets([tp])[tp]  # high watermark: next produced offset
print(f"readable offsets: {first_offset} .. {next_offset - 1}\n")


def read_from(offset, n=2):
    """Seek to `offset`, then print the next `n` messages."""
    explorer.seek(tp, offset)
    seen = 0
    while seen < n:
        batch = explorer.poll(timeout_ms=2000, max_records=n - seen)
        if not batch:
            break
        for record in batch[tp]:
            print(f"  offset {record.offset}: {loads(record.value.decode('utf-8'))}")
            seen += 1


# Three start offsets across the valid range: start, midpoint, last 2 messages.
start_offsets = [
    first_offset,
    (first_offset + next_offset) // 2,
    max(first_offset, next_offset - 2),
]

for start in start_offsets:
    print(f"--- seek to offset {start} ---")
    read_from(start, 2)

explorer.close()

# auto_offset_reset could reach offset 0 ('earliest') and the high watermark
# ('latest') on its own. Offset 10 (the midpoint) needed seek().

readable offsets: 0 .. 19

--- seek to offset 0 ---
  offset 0: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:17', 'temperature_f': 82}
  offset 1: {'city': 'Pittsburgh', 'timestamp': '2026-08-27 18:45:18', 'temperature_f': 64}
--- seek to offset 10 ---
  offset 10: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:23', 'temperature_f': 82}
  offset 11: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:23', 'temperature_f': 82}
--- seek to offset 18 ---
  offset 18: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:27', 'temperature_f': 82}
  offset 19: {'city': 'Shanghai', 'timestamp': '2026-08-27 18:45:27', 'temperature_f': 82}


# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat.

Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html


In [ ]:
# Commands used (run these in a terminal, not as Python):
#
# 1. First 5 messages from the earliest offset, showing each offset:
# kcat -b localhost:9092 -t lab01-helenlwang -C -o beginning -c 5 -e -f "%o: %s\n"
#
# 2. Same command from an absolute offset in the middle of the range:
# kcat -b localhost:9092 -t lab01-helenlwang -C -o 8 -c 5 -e -f "%o: %s\n"
#
# Flags:
# -b broker   -t topic   -C consume   -o start offset
# -c count    -e exit at end of log   -f print format (%o offset, %s payload)
print("see markdown cell below for captured output")

### Your kcat output

**Command 1** — from the beginning (`-o beginning -c 5`):
```
0: {"city": "Shanghai", "timestamp": "2026-08-27 18:45:17", "temperature_f": 82}
1: {"city": "Pittsburgh", "timestamp": "2026-08-27 18:45:18", "temperature_f": 64}
2: {"city": "Pittsburgh", "timestamp": "2026-08-27 18:45:18", "temperature_f": 64}
3: {"city": "Shanghai", "timestamp": "2026-08-27 18:45:19", "temperature_f": 82}
4: {"city": "Shanghai", "timestamp": "2026-08-27 18:45:19", "temperature_f": 82}
```

**Command 2** — from absolute offset 8 (`-o 8 -c 5`):
```
8: {"city": "Pittsburgh", "timestamp": "2026-08-27 18:45:22", "temperature_f": 64}
9: {"city": "Shanghai", "timestamp": "2026-08-27 18:45:22", "temperature_f": 82}
10: {"city": "Shanghai", "timestamp": "2026-08-27 18:45:23", "temperature_f": 82}
11: {"city": "Shanghai", "timestamp": "2026-08-27 18:45:23", "temperature_f": 82}
12: {"city": "Shanghai", "timestamp": "2026-08-27 18:45:24", "temperature_f": 82}
```

These offsets match the Python consumer: message 8 is Pittsburgh 64, message 10 is Shanghai 82.

**What does the offset refer to?**

An offset is a monotonically increasing integer that is the address of a message inside one partition's log. Offset 0 is the oldest still-retained message. Offset 8 is the 9th message. If a consumer disconnects after committing offset 8, it resumes at 9 and does not re-read 0–8 (unless you seek). `auto_offset_reset` cannot start at 8; only `seek()` / `kcat -o 8` can.


### Optional: peek at the movie stream (group project)

```
kcat -b localhost:9092 -L
kcat -b localhost:9092 -t movielog1 -C -o beginning -c 3 -e -f "%o: %s\n"
```

Sample `movielog1` lines (userid is a number, movieid is a string):

```
0: 2026-07-17T14:50:46,28083,GET /data/m/ace+ventura+pet+detective+1994/0.mpg
91839544: 2026-08-27T18:44:45.585,118511,GET /data/m/fight+club+1999/19.mpg
```

Then look up metadata:

```
curl http://128.2.220.123:8080/movie/fight+club+1999
curl http://128.2.220.123:8080/user/118511
```

Before the group project starts, read `movielog1`. After teams are assigned, use `movielogN` for your team. Please only write to Kafka for this lab.
